# E4 · Adaptación LoRA de CLIP sobre E3-100

**Objetivo.** Medir el beneficio adicional de adaptar el encoder visual después de E3.

**Pregunta.** ¿LoRA en la atención de CLIP mejora la recuperación frente a E3-100 sin perder transferencia a PetTrace?

**Hipótesis.** Una actualización de bajo rango puede representar señales de identidad que la MLP sobre embeddings congelados no captura.

| Elemento | Diseño fijado |
|---|---|
| Intervención | LoRA `r=8`, `alpha=16` en `q_proj` y `v_proj` de CLIP ViT-B/32 |
| Congelado | YOLO, pesos base/proyección de CLIP y cabeza E3-100 |
| Control | E3-100 semilla 46, seleccionada entre 45–49 solo por validación |
| Muestra | Fuentes públicas refinadas `v2`, equivalentes a `pettrace-training@v4.100`; PetTrace no participa del entrenamiento |
| Selección | mAP macro por especie en validación pública; nunca por test |
| Evaluación | Test público v2 y PetTrace v6, con galerías separadas |
| Estado | **Reproducción E4 sobre PetTrace v6; resultados pendientes** |

> Las salidas conservadas abajo corresponden a la corrida anterior sobre v5 y se mantienen como evidencia. Al ejecutar el cuaderno serán reemplazadas por resultados y recibos de v6.


### Entorno

En una T4 nueva, esta única instalación obtiene el código exacto desde PyPI. Las credenciales se leen de Colab Secrets y nunca se imprimen.

In [1]:
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks,tracking]==0.2.39"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import json, os, sys
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import numpy as np
import torch
from IPython.display import Markdown, display
from google.colab import userdata
import pettrace
from pettrace.tracking import PRODUCTION_URI, source_identity

if not (3, 11) <= sys.version_info[:2] < (3, 14) or sys.platform != "linux":
    raise RuntimeError("E4 requires Colab Linux with Python 3.11–3.13")
if pettrace.__version__ != "0.2.39":
    raise RuntimeError("Install PetTrace 0.2.39 before continuing")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; a T4 is sufficient")
print("Access [1/2] Loading Colab Secrets…")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise TypeError(f"Colab Secret {name} must be a non-empty text value")
    os.environ[name] = value
print("Access [2/2] Credentials available.")
EXPERIMENT, DATASET = "E4", "v6"
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
TRAINING_DATASET_VERSION, SOURCE_SAMPLE_PCT, SEED = "v2", 100, 46
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
BASE_MODEL = "models:/pettrace-embedder/2"
LOG_TO_MLFLOW = True
REGISTER_E4 = True
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
REPORT = Path("/content/reports") / "e4_v6" / RUN_LABEL
REPORT.mkdir(parents=True)
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
print("Package:", pettrace.__version__, "| GPU:", torch.cuda.get_device_name(0), "| Report:", REPORT)

Access [1/2] Loading Colab Secrets…
Access [2/2] Credentials available.
Package: 0.2.38 | GPU: Tesla T4 | Report: /content/reports/e4_v5/20261109T052016Z-0394820a


## Fase 1. Preparación y preprocesamiento de datos

E4 **carga** las fuentes refinadas públicas `v2` desde commits inmutables de Hugging Face; no adquiere ZIP, no limpia y no ejecuta YOLO. Sus imágenes, identidades y splits son exactamente los de `v4.100`; `v2` añade únicamente las cajas verificadas que reproducen el recorte E2.

In [3]:
from pettrace.data import training
from pettrace.data.volume import training_volume

print("Dataset [1/2] Downloading pinned public v2 shards and verifying hashes…")
prepared = training(version=TRAINING_DATASET_VERSION,
    cache_dir="/content/datasets/pettrace-refined", token=os.environ["HF_TOKEN"])
manifest_rows = prepared.rows
training_source = prepared.provenance
coverage_metrics, coverage_params = training_volume(prepared, SOURCE_SAMPLE_PCT, REPORT)
(REPORT / "transport-equivalence.json").write_text(json.dumps(training_source, indent=2))
print("Dataset [2/2] Verified:", prepared.fingerprint, "| photos:", len(manifest_rows),
      "| identities:", prepared.summary["retained_identities"],
      "| commits:", [source.commit[:12] for source in prepared.sources])

Dataset [1/2] Downloading pinned public v2 shards and verifying hashes…
Dataset: verifying and materializing alessandro54/labeled-cats-in-the-wild-refined@581e44a04fd4…
Dataset: lcw ready (131,975 photos).
Dataset: verifying and materializing alessandro54/dogs-of-the-world-refined@9fdd6e5f7eaa…
Dataset: dogs-world ready (61,851 photos).
Dataset [2/2] Verified: d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58 | photos: 193826 | identities: 42113 | commits: ['581e44a04fd4', '9fdd6e5f7eaa']


In [4]:
from pettrace.tracking import production_context
from pettrace.experiments.e3 import load_benchmarks
production_context()
benchmarks = load_benchmarks(prepared, REPORT, private_version=DATASET, private_commit=DATASET_COMMIT)
private_ds, public_ds, PUBLIC_REPORT = benchmarks.private, benchmarks.public, benchmarks.public_report
print("Public test:", len(public_ds.photos), "photos | PetTrace:", len(private_ds.photos), "photos")

Loading immutable benchmarks:   0%|          | 0/3 [00:00<?, ?step/s]

Benchmarks ready in 23.4s: 277 public · 425 PetTrace.
Public test: 277 photos | PetTrace: 425 photos


## Fase 2. Representación y entrenamiento

Se descarga la cabeza E3-100 del run fijado y se verifica su SHA-256. CLIP usa la revisión inmutable indicada arriba. Solo los adaptadores LoRA son entrenables; antes de ajustar se comprueba que su inicialización reproduce E3.

LoRA sigue a [Hu et al. (2022)](https://arxiv.org/abs/2106.09685). La pérdida batch-hard sigue a [Hermans et al. (2017)](https://arxiv.org/abs/1703.07737). Capas, rango, presupuesto y control son decisiones de esta investigación.

In [5]:
from pettrace.evaluation.core import load_model
from pettrace.experiments.e4 import load_control_head
from pettrace.training.lora import Config, PreparedPreprocessor, build_encoder, control_equivalence

base_model = load_model(BASE_MODEL)
if not base_model.golden_ok or base_model.pipeline.detector is None:
    raise RuntimeError("The fixed E2 base failed verification")

head, control_receipt = load_control_head(REPORT)
preprocessor = PreparedPreprocessor(base_model.pipeline, prepared)

training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
train_rows = [row for row in training_rows if row["split"] == "train"]
gate = [
    next(row for row in sorted(train_rows, key=lambda item: item["sha256"])
         if row["species"] == species)
    for species in ("cat", "dog")
]

checked = preprocessor.validate(gate)
pixels = preprocessor.pixels(gate)
e2 = base_model.pipeline.session.run(
    [base_model.pipeline.output_name],
    {base_model.pipeline.input_name: pixels},
)[0]

config = Config(seed=SEED)
encoder, adapted_layers = build_encoder(head, CLIP_REVISION, config=config)
encoder = encoder.to("cuda")  # Move the newly injected LoRA parameters to the GPU.

parity = control_equivalence(encoder, pixels, e2)
(REPORT / "control-equivalence.json").write_text(json.dumps(
    {
        **parity,
        "cached_crop_photos": checked,
        "adapted_layers": adapted_layers,
    },
    indent=2,
))

print("Control parity:", parity, "| LoRA layers:", len(adapted_layers))

Control parity: {'minimum_cosine': 0.9999999403953552, 'maximum_absolute_error': 1.955777406692505e-07, 'photos': 2} | LoRA layers: 24


In [6]:
from pettrace.training.lora import fit, trainable_parameters
from pettrace.ui import markdown_table

run_config = {"experiment": EXPERIMENT, "dataset": TRAINING_DATASET_VERSION,
              "dataset_fingerprint": prepared.fingerprint, "clip_revision": CLIP_REVISION,
              "control": control_receipt, "training": asdict(config),
              "trainable_parameters": trainable_parameters(encoder),
              "log_to_mlflow": LOG_TO_MLFLOW, "register_model": REGISTER_E4}
(REPORT / "training-config.json").write_text(json.dumps(run_config, indent=2))
encoder, training_summary = fit(encoder, training_rows, preprocessor, REPORT,
                                config=config, device="cuda")
(REPORT / "training-summary.json").write_text(json.dumps(training_summary, indent=2))
display(Markdown("### Checkpoint seleccionado por validación\n\n" + markdown_table(
    ["Seed", "Mejor época", "mAP macro de validación"],
    [[SEED, training_summary["best_epoch"], f"{training_summary['best_val_mAP']:.4f}"]],
)))

Training seed 46:   0%|          | 0/30 [00:00<?, ?epoch/s]

### Checkpoint seleccionado por validación

| Seed | Mejor época | mAP macro de validación |
|---|---|---|
| 46 | 29 | 0.8454 |

In [7]:
from pettrace.training.lora import export_encoder
model_dir = REPORT / "model"
onnx_hashes = export_encoder(base_model.dir, encoder, model_dir,
    model_version="clip-vit-b32-crop-lora-e3-v4.100@v3", verification_pixels=pixels)
e4_model = load_model(str(model_dir))
if not e4_model.golden_ok:
    raise RuntimeError("Exported E4 model failed golden verification")
print("Exported:", onnx_hashes)

/usr/local/lib/python3.13/dist-packages/transformers/models/clip/modeling_clip.py:243: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not interpolate_pos_encoding and (height != self.image_size or width != self.image_size):


Exported: {'detector.onnx': 'da8f9b9ebea5157cdcefa7bea9fd782249ebdf119f917c07d64e5b41942b9c9d', 'encoder.onnx': '84e7ae17b8639f97e6642158c4f3f2d5d8852b87dd7ac239eefdf0b9ad7d9550'}


## Fase 3. Evaluación y análisis

El checkpoint ya quedó fijado por validación. Ahora se evalúan, por separado, el test público v2 y PetTrace v6. El modelo E3 congelado se reevalúa sobre esas mismas consultas y galerías para mantener un control comparable. Rank-k usa el máximo score por identidad y mAP ordena fotografías en una galería cerrada de la misma especie. La EER es descriptiva porque se calcula sobre los mismos pares evaluados.

**Limitaciones.** Los benchmarks ya fueron observados en experimentos anteriores; no garantizan separación entre sesiones, búsqueda abierta ni generalización poblacional. Esta corrida exploratoria usa una sola semilla y no estima variabilidad de optimización.

In [8]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import metrics_table, save
from pettrace.evaluation.research import save_benchmark_comparison
from pettrace.experiments.e4 import CONTROL_MODEL_VERSION
from pettrace.tracking import save_provenance

public_ev = evaluate(public_ds, e4_model.pipeline, zero_shot=True)
private_ev = evaluate(private_ds, e4_model.pipeline, zero_shot=True)
public_summary = save(public_ev, e4_model, str(PUBLIC_REPORT))
private_summary = save(private_ev, e4_model, str(REPORT))
save_provenance(PUBLIC_REPORT, e4_model); save_provenance(REPORT, e4_model)
control_model = load_model(f"models:/pettrace-embedder/{CONTROL_MODEL_VERSION}")
if not control_model.golden_ok:
    raise RuntimeError("The frozen E3 control failed golden verification")
public_e3 = evaluate(public_ds, control_model.pipeline, zero_shot=True)
private_e3 = evaluate(private_ds, control_model.pipeline, zero_shot=True)
public_e3_report = PUBLIC_REPORT / "baseline_e3"
private_e3_report = REPORT / "baseline_e3"
public_e3_summary = save(public_e3, control_model, str(public_e3_report), label="E3-100 · Public v2")
private_e3_summary = save(private_e3, control_model, str(private_e3_report), label="E3-100 · PetTrace v6")
save_provenance(public_e3_report, control_model); save_provenance(private_e3_report, control_model)
(REPORT / "e3-control-v6.json").write_text(json.dumps({
    **control_receipt,
    "model_uri": f"models:/pettrace-embedder/{CONTROL_MODEL_VERSION}",
    "public_dataset_commit": public_ds.commit,
    "private_dataset_commit": private_ds.commit,
}, indent=2))
display(Markdown("### Test público\n" + metrics_table({"E3-100": public_e3_summary, "E4": public_summary})))
display(Markdown("### PetTrace v6\n" + metrics_table({"E3-100": private_e3_summary, "E4": private_summary})))
domain_table, _ = save_benchmark_comparison(public_ev, private_ev, REPORT, EXPERIMENT)
display(Markdown("### Dominios separados\n" + domain_table))
comparison = {domain: {metric: current["global"][metric] - control["global"][metric]
               for metric in ("rank1", "rank2", "mAP")}
              for domain, current, control in (("public", public_summary, public_e3_summary),
                                                ("private", private_summary, private_e3_summary))}
(REPORT / "comparison.json").write_text(json.dumps(comparison, indent=2))
print("E4 − E3:", comparison)

### Test público
| Metric | E3-100 | E4 |
|---|---|---|
| Queries / gallery per species | 277 / 34 animals | 277 / 34 animals |
| Global Rank-1 (chance 2.9 %) | **81.6 %** | **87.0 %** |
| Rank-1 cats / dogs | 77.2 % / 87.8 % | 87.7 % / 86.1 % |
| Global Rank-5 (chance 14.7 %) | 98.9 % | 99.6 % |
| Global mAP | 0.767 | 0.837 |
| mAP cats / dogs | 0.701 / 0.861 | 0.808 / 0.878 |
| EER / threshold | 9.3 % / 0.834 | 7.4 % / 0.775 |
| Minimum similarity, same animal | 0.690 | 0.555 |
| Maximum similarity, different animal | 0.982 | 0.975 |

### PetTrace v5
| Metric | E3-100 | E4 |
|---|---|---|
| Queries / gallery per species | 425 / 25 animals | 425 / 25 animals |
| Global Rank-1 (chance 4.0 %) | **75.5 %** | **82.1 %** |
| Rank-1 cats / dogs | 80.5 % / 72.3 % | 85.8 % / 79.7 % |
| Global Rank-5 (chance 20.0 %) | 99.1 % | 99.8 % |
| Global mAP | 0.678 | 0.725 |
| mAP cats / dogs | 0.696 / 0.667 | 0.760 / 0.702 |
| EER / threshold | 11.5 % / 0.797 | 10.1 % / 0.770 |
| Minimum similarity, same animal | 0.456 | 0.270 |
| Maximum similarity, different animal | 1.000 | 1.000 |

### Dominios separados
| Dataset | Group | Queries | Gallery (identities) | Rank-1 | Rank-2 | mAP |
|---|---|---:|---:|---:|---:|---:|
| Public v2 · test | global | 277 | 34 | 87.00% | 96.03% | 0.8372 |
| Public v2 · test | cat | 162 | 34 | 87.65% | 95.68% | 0.8079 |
| Public v2 · test | dog | 115 | 26 | 86.09% | 96.52% | 0.8783 |
| PetTrace v5 | global | 425 | 25 | 82.12% | 94.82% | 0.7251 |
| PetTrace v5 | cat | 169 | 17 | 85.80% | 94.08% | 0.7597 |
| PetTrace v5 | dog | 256 | 25 | 79.69% | 95.31% | 0.7022 |

E4 − E3: {'public': {'rank1': 0.054151624548736454, 'rank2': 0.03971119133574008, 'mAP': 0.0699635500396848}, 'private': {'rank1': 0.0658823529411765, 'rank2': 0.014117647058823568, 'mAP': 0.046582544199721965}}


### Publicación obligatoria

La corrida solo termina cuando ambos dominios quedan publicados. Mientras el runtime siga activo, un error de publicación conserva el reporte local y permite reintentar esta celda sin volver a entrenar. MLflow conserva la evidencia publicada y el modelo candidato; el registro no mueve aliases.

In [9]:
from pettrace.experiments.e3 import publish_results, register_candidate
publication = publish_results(public_ev, private_ev, e4_model, REPORT, PUBLIC_REPORT, SOURCE_SHA256,
    experiment=EXPERIMENT, source_sample_pct=SOURCE_SAMPLE_PCT, config=config,
    training_summary=training_summary, prepared_fingerprint=prepared.fingerprint,
    training_source=training_source, coverage_metrics=coverage_metrics,
    coverage_params=coverage_params, private_dataset_version=DATASET)
run_id, public_run_id = publication["run_id"], publication["public_run_id"]
if REGISTER_E4:
    registration = register_candidate(REPORT, run_id, public_run_id, experiment=EXPERIMENT)
    print("Candidate version:", registration["version"], "| no alias promoted")
print("E4 complete · private:", run_id, "| public:", public_run_id)

🏃 View run E4 · clip-vit-b32-crop-lora-e3-v4.100@v3 on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/5eec13717d7345fbb348a5cb95ae42c1
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3
🏃 View run E4 · clip-vit-b32-crop-lora-e3-v4.100@v3 on pettrace-reid@v5 at: https://ml.pettrace.app/#/experiments/3/runs/b2dc5966796d46d4ac9a68e1b5a4b336
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


2026/11/09 06:22:14 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: pettrace-embedder, version 22


Candidate version: 22 | no alias promoted
E4 complete · private: b2dc5966796d46d4ac9a68e1b5a4b336 | public: 5eec13717d7345fbb348a5cb95ae42c1
